# Automatic Differentiation

Research on the topic of Automatic Differentiation and Backpropagation with `torch.autograd`.

In this Backpropagation, parameters (model weights) are adjusted according to the gradient of the loss function with respect to the given parameter.

`torch.autograd` supports automatic computation of gradient for any computational graph.


# Notebook Setup

## Imports

In [2]:
# Import Standard Libraries
import torch

# Tensors

## Basic Example

In [2]:
# Input and expected output
x = torch.ones(5)
y = torch.zeros(3)

# Parameters
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)

# Feed forward
z = torch.matmul(x, w)+b

# Compute loss
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

`grad_fn` is a reference to the Backpropagation function.

In [3]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x107eff970>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x107eff940>


The function `backward()` of the loss function compute the gradients of the model's parameters.

In [4]:
# Compute the gradients of the model's parameters
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.2574, 0.0718, 0.0215],
        [0.2574, 0.0718, 0.0215],
        [0.2574, 0.0718, 0.0215],
        [0.2574, 0.0718, 0.0215],
        [0.2574, 0.0718, 0.0215]])
tensor([0.2574, 0.0718, 0.0215])


The `requires_grad` is used to track the history of the computations and support gradients computation. Sometimes it's not required and it is possible to stop it.

In [5]:
# With computation tracking
z = torch.matmul(x, w)+b
print(z.requires_grad)

# Without computation tracking
with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

True
False


In [6]:
# Same effect but with 'detach' function
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

False


## Gradient Function

The Gradient Function `grad_fn` keeps track of what operation lead to the creation of that specific tensor. Let's see some examples.

In [8]:
tensor_1 = torch.linspace(1, 10, steps=25, requires_grad=True)

print('Initial Tensor')
print(tensor_1)

tensor_2 = torch.sin(tensor_1)

print()
print('Sin(tensor) - Gradient Function')
print(tensor_2.grad_fn)

tensor_3 = 2 * tensor_2

print()
print('2 * Sin(tensor) - Gradient Function')
print(tensor_3.grad_fn)

print()
print('Gradient Function History')
print(tensor_3.grad_fn)
print(tensor_3.grad_fn.next_functions)

Initial Tensor
tensor([ 1.0000,  1.3750,  1.7500,  2.1250,  2.5000,  2.8750,  3.2500,  3.6250,
         4.0000,  4.3750,  4.7500,  5.1250,  5.5000,  5.8750,  6.2500,  6.6250,
         7.0000,  7.3750,  7.7500,  8.1250,  8.5000,  8.8750,  9.2500,  9.6250,
        10.0000], requires_grad=True)

Sin(tensor) - Gradient Function

2 * Sin(tensor) - Gradient Function

Gradient Function History
((<SinBackward0 object at 0x10c503880>, 0), (None, 0))


Notice how we can keep track of the history of the transformation happened to the tensor through the `grad_fn` attribute.

In [16]:
# It is always possible to turn off the automatic gradient differentiation
a = torch.ones(2, 3, requires_grad=True)
print(a)

b1 = 2 * a
print(b1)

a.requires_grad = False
b2 = 2 * a
print(b2) # Note how the grad_fn is not present for this operation

tensor([[1., 1., 1.],
        [1., 1., 1.]], requires_grad=True)
tensor([[2., 2., 2.],
        [2., 2., 2.]], grad_fn=<MulBackward0>)
tensor([[2., 2., 2.],
        [2., 2., 2.]])


# Computational Graph

## Definition
Conceptually, autograd keeps a record of data (tensors) and all executed operations (along with the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function objects. In this DAG, leaves are the input tensors, roots are the output tensors. By tracing this graph from roots to leaves, you can automatically compute the gradients using the chain rule.

## Feed Forward Step
Two operations at the same time:
- Compute the resulting tensor
- Store in the DAG the operation's gradient function (`grad_fn`)

## Backward
This operation is called over the root of the DAG. The `autograd` then does:
- Compute the gradient through `.grad_fn`
- Accumulate them in `.grad` attribute
- Using the chain rule, propagates all the way to the leaf tensors

## Example

In [9]:
BATCH_SIZE = 16
DIM_IN = 1000
HIDDEN_SIZE = 100
DIM_OUT = 10

class TinyModel(torch.nn.Module):
    """Create just a tiny model to see the Backpropagation process"""
    def __init__(self):
        super(TinyModel, self).__init__()

        self.layer1 = torch.nn.Linear(DIM_IN, HIDDEN_SIZE)
        self.relu = torch.nn.ReLU()
        self.layer2 = torch.nn.Linear(HIDDEN_SIZE, DIM_OUT)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# Create some sample input and ideal output
some_input = torch.randn(BATCH_SIZE, DIM_IN, requires_grad=False)
ideal_output = torch.randn(BATCH_SIZE, DIM_OUT, requires_grad=False)

model = TinyModel()

In [10]:
# Have a look at the initial weights and gradients (ofc they are zero now)
print(model.layer2.weight[0][0:10]) # just a small slice
print(model.layer2.weight.grad)

tensor([-0.0497,  0.0865, -0.0707,  0.0845,  0.0598,  0.0294, -0.0636,  0.0508,
         0.0736, -0.1000], grad_fn=<SliceBackward0>)
None


In [11]:
# Fake a feed forward step
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
prediction = model(some_input)
loss = (ideal_output - prediction).pow(2).sum()
print(loss)

tensor(159.7465, grad_fn=<SumBackward0>)


In [12]:
# Call the backward function to compute the gradients
# NOTE: See how the weights are not yet updated, but the gradients have been changed
loss.backward()
print(model.layer2.weight[0][0:10])
print(model.layer2.weight.grad[0][0:10])

tensor([-0.0497,  0.0865, -0.0707,  0.0845,  0.0598,  0.0294, -0.0636,  0.0508,
         0.0736, -0.1000], grad_fn=<SliceBackward0>)
tensor([ 0.2930,  0.6573, -0.1068,  0.5366, -2.3651, -1.5714, -0.5419, -0.5545,
         0.8112,  2.3917])


In [13]:
# Now update the weights with the computed gradients
optimizer.step()
print(model.layer2.weight[0][0:10])
print(model.layer2.weight.grad[0][0:10])

tensor([-0.0500,  0.0859, -0.0706,  0.0840,  0.0622,  0.0310, -0.0630,  0.0513,
         0.0728, -0.1024], grad_fn=<SliceBackward0>)
tensor([ 0.2930,  0.6573, -0.1068,  0.5366, -2.3651, -1.5714, -0.5419, -0.5545,
         0.8112,  2.3917])


In [15]:
# Remember to call the `zero_grad()` in order to reset the gradients
optimizer.zero_grad()
print(model.layer2.weight.grad)

None
